In [166]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import du processeur de production spécialisé
from tools.OI_class_OP import OI_ProductionProcessor
from tools.OI_Dashboard import ProductionDashboard
from tools.OI_Dashboard_v2 import AjouterVisualisationsAvancees


# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [167]:
# Cellule 2 : Définition des métadonnées de tags API
tags = [
    {'tag':'WQ33222VA', 'nom':'ester_cons','info':'Totalisation du Peson Acetate/Propionate', 'agg': 'FIRST' },
    {'tag':'NOP_ESTERS', 'nom':'ester_nop','info':"Nombres des opérations d'esters",'agg':'FIRST' },
    {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate','agg':'FIRST' },
    {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate', 'agg': 'MEAN'},
    {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate', 'agg': 'MEAN'},
    {'tag':'PU3310VA_Sign','nom':'PU3310','info':'signature du PU3310','agg':'FIRST'},
    {'tag':'PU3320VA_Sign','nom':'PU3320','info':'signature du PU3320','agg':'FIRST'},
    {'tag':'PU3340VA_Sign','nom':'PU3340','info':'signature du PU3340','agg':'FIRST'},
    {'tag':'FQ32202VA_UV','nom':'Hexane','info':'VA diluée dans de l\'hexane', 'agg': 'FIRST'},
    {'tag':'CTY_A3230A_Teneur en rétinol', 'nom':'Retinol_UV','info':'A3230A teneur en rétinol lavé', 'agg': 'MEAN'},
    {'tag':'LI33203VA','nom':'R33020','info':'niveau du R33020', 'agg': 'FIRST'},
    {'tag':'LI33218VA','nom':'R33022','info':'niveau du R33022', 'agg': 'FIRST'},
    {'tag':'LI33225VA','nom':'R33061','info':'niveau du R33061', 'agg': 'FIRST'},
    {'tag':'WI33222VA','nom':'R33060','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'FQ32202VA','nom':'retinol_cons','info':'peson du R33060', 'agg': 'FIRST'},
    {'tag':'LI32209VA','nom':'R32031','info':'niveau du R32031', 'agg': 'FIRST'},
]

In [168]:
# Cellule 3 : Variable Produits unifiée (Acetate & Propionate)
# Plus aucune distinction batch / continu pour le calcul global du stock d'un produit.
produits = [
    {
        'nom': 'Ester',
        'conso': {
            'value': 'ester_cons',
            'scale': 1e-9,
            'type':'A/P',
            'uv': ['Acetate_UV', 'Propionate_UV'],
        },
        'CMJ': 9.5,
        'NOP': {
            'value':'NOP_ESTERS',
            'scale': 1,
            'type': None,
        },
        'stock': [
            # Batchs
            {'pu': 'PU3310', 'in': 540, 'out': 710, 'value': 'Hexane', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
            {'pu': 'PU3310', 'in': 710, 'out': 2320, 'value': None, 'uv': None, 'scale': 2.71},
            {'pu': 'PU3320', 'in': 430, 'out': 2020, 'value': None, 'uv': None, 'scale': 2.71},
            # Continus
            {'pu': None, 'value': 'R33020', 'min': 14, 'epalage': [[28.62, 21.22, 3.866, -0.0983], [-228.84, 74.557]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33022', 'min': 14, 'epalage': [[6.25, 5.4525, 0.898, -0.0256], [-35, 97, 16.039]], 'uv': 'Retinol_UV', 'scale': 1/100., 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': None, 'value': 'R33061', 'min': 18, 'epalage': [[0.69, 0.72, 0.296, -0.0059], [-30.03, 5.836]], 'uv': None, 'scale': 0.95 , 'cond':'A/P', 'val_cond':[1/344,1/359]},
            {'pu': 'PU3340', 'in': 0, 'out': 710, 'value': 'R33060', 'uv': None, 'scale': 0.95, 'cond': 'A/P', 'val_cond': [1/344, 1/359]}
        ]
    },
    {
        'nom': 'Retinol',
        'conso': {
            'value':'retinol_cons',
            'scale': 1e-5, # Ajustement de l'échelle pour le retinol g et analyse en %
            'uv': ['Retinol_UV','Retinol_UV'],
        },
        'CMJ': 1,
        'stock': [
            # Batchs
            {'pu': None, 'value': 'R32031', 'uv': 'Retinol_UV', 'scale': 1/(825 * 100.)},
        ]
    }
]

In [169]:
# Cellule 4 : Initialisation du processeur de production spécialisé
processor = OI_ProductionProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start = '2026-01-01',
    end = '2026-12-31',
    tags_metadata = tags,
    produits = produits,
    interval = 'PT10M',
    verbose = False
)

In [170]:
(processor.agg_mapping)

{'WQ33222VA': 'FIRST',
 'NOP_ESTERS': 'FIRST',
 '3340_type': 'FIRST',
 'CTY_ACV43A_Teneur Vit. A (UV)': 'MEAN',
 'CTY_A3340FGB_Teneur arr. Vit. A (UV)': 'MEAN',
 'PU3310VA_Sign': 'FIRST',
 'PU3320VA_Sign': 'FIRST',
 'PU3340VA_Sign': 'FIRST',
 'FQ32202VA_UV': 'FIRST',
 'CTY_A3230A_Teneur en rétinol': 'MEAN',
 'LI33203VA': 'FIRST',
 'LI33218VA': 'FIRST',
 'LI33225VA': 'FIRST',
 'WI33222VA': 'FIRST',
 'FQ32202VA': 'FIRST',
 'LI32209VA': 'FIRST'}

In [171]:
# Cellule 5 : Téléchargement et calcul automatique des bilans par produit
processor.merge()
processor.compute_production_balance()

# Visualisation des premières lignes calculées
processor.data.describe()

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester,consommation_Retinol,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
count,2.304300e+04,23043.000000,23043.000000,2.304300e+04,2.304300e+04,23031.000000,23034.000000,23030.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000,23043.000000
mean,2.465305e+06,1973.724434,0.072098,2.226375e+06,2.276202e+06,1030.593982,963.968612,460.805037,2610.936368,20.778915,46.250404,4.954866,7.236006,688.970153,480323.698339,47.798198,543.956817,6.503712,426021.475214,0.012037,543.956817,0.917425,544.874241,426021.475214,0.000482,426021.475696
std,1.530607e+05,125.535402,0.258637,3.760872e+04,5.413027e+04,622.683904,553.927926,119.068948,1880.464161,0.362300,20.422216,2.473153,11.498947,353.419598,255507.004521,21.055398,342.656396,2.158374,493825.156797,0.005304,342.656396,2.158374,343.280184,493825.156797,0.005304,493825.157091
min,2.221310e+06,1774.000000,0.000000,2.078915e+06,2.188000e+06,100.000000,100.000000,100.000000,0.000000,20.000000,-8.417280,-0.695313,-0.536877,4.790160,480.250000,-1.354620,0.000000,0.702392,0.000000,-0.000342,0.000000,-4.883895,-1.990054,0.000000,-0.011897,0.000000
25%,2.319790e+06,1855.000000,0.000000,2.220809e+06,2.242000e+06,400.000000,400.000000,410.000000,0.000000,20.600000,24.427050,3.063825,2.240025,424.412000,261830.000000,39.844700,217.947562,5.393334,68.752013,0.010003,217.947562,-0.192954,217.437819,68.752013,-0.001552,68.750781
50%,2.453780e+06,1962.000000,0.000000,2.231260e+06,2.300000e+06,1310.000000,1110.000000,410.000000,3961.980000,20.800000,46.080800,5.988280,3.015630,685.019000,458717.000000,45.102200,516.339330,6.792363,161.843939,0.011513,516.339330,1.206076,516.448547,161.843939,-0.000042,161.843536
75%,2.595120e+06,2080.000000,0.000000,2.244406e+06,2.314000e+06,1310.000000,1555.000000,410.000000,4043.440000,21.200000,65.045250,6.558405,4.035160,949.348000,705011.500000,67.183600,834.062849,7.984893,998520.821515,0.016849,834.062849,2.398606,836.397228,998520.821515,0.005294,998520.822781
max,2.750770e+06,2207.000000,1.000000,2.325113e+06,2.626000e+06,2340.000000,2290.000000,810.000000,4836.070000,21.200000,91.609100,12.334800,74.049600,1322.300000,998943.000000,79.734400,1184.358817,12.323393,998626.320437,0.019909,1184.358817,6.737105,1187.714779,998626.320437,0.008354,998626.327418


In [172]:
# ✨ INITIALISER LE DASHBOARD ✨
dashboard = ProductionDashboard(processor)
AjouterVisualisationsAvancees(dashboard) 

print("\n✓ Dashboard prêt pour utilisation!")
#dashboard.resume_complet()


✓ Dashboard initialisé
  Produits: Ester, Retinol
  Période: 2026-01-01 → 2026-06-10
✅ Visualisations avancées ajoutées au dashboard!

   Nouvelles méthodes disponibles:
   • dashboard.plot_histogramme_tous_produits(mois=3)
   • dashboard.plot_waterfall_mois(mois=3)
   • dashboard.plot_histogramme_jours_mois_v1(mois=3, nom_produit='Ester')
   • dashboard.plot_histogramme_jours_mois_v2(mois=3, nom_produit='Ester')


✓ Dashboard prêt pour utilisation!


In [173]:
# Voir l'évolution temporelle d'un produit
dashboard.afficher_bilan_produit('Ester')


In [174]:
dashboard.plot_histogramme_jours_mois_v2(mois=6, annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.3908
  Variation de Stock     : 0.0057
  Stock Entrée.          : -0.0004
  Stock Sortie.          : 0.0052
  Production             : 3.3964
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JUIN 2026 (Termin


  📊 STATISTIQUES - PRODUCTION ESTER PAR JOUR - JUNE 2026
Nombre de jours complets:           10
Production moyenne:                 9.28
Production min/max:                 6.20 / 14.13
Écart-type:                         2.19
Coefficient de variation:           23.6%
Production totale mois:             92.82
TRS mois: (jours complets)          9.28
Jours au-dessus de la moyenne:      5 / 10

🎯 RATIO OOE (Production / CMJ en %):
  CMJ (Cible Journalière):            9.50
  OOE Moyen (par jour):               97.7%
  OOE Min/Max (par jour):             65.3% / 148.7%
  OOE Cumulé à date:                  97.7%
  Jours > 100%:                       4 / 10

  Détail OOE par jour:
    J01:  113.6% ✅
    J02:   85.6% ⚠️ 
    J03:   94.0% ⚠️ 
    J04:  113.8% ✅
    J05:   77.9% ❌
    J06:  148.7% ✅
    J07:   99.1% ⚠️ 
    J08:   74.7% ❌
    J09:  104.3% ✅
    J10:   65.3% ❌



In [175]:
debut = '06-06-2026 02:00:00'
fin = '07-06-2026 02:00:00'
processor.data[debut:fin]

,ester_cons,ester_nop,A/P,Acetate_UV,Propionate_UV,PU3310,PU3320,PU3340,Hexane,Retinol_UV,R33020,R33022,R33061,R33060,retinol_cons,R32031,consommation_Ester,stock_Ester,consommation_Retinol,stock_Retinol,conso_delta_Ester,delta_stock_Ester,production_Ester,conso_delta_Retinol,delta_stock_Retinol,production_Retinol
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-06-06 02:00:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,430.0,410.0,4042.420,20.6,64.3027,0.32374,3.30051,465.954,787465.0,37.6188,1139.793007,6.752352,998613.508267,0.009393,1139.793007,1.166064,1140.959071,998613.508267,-0.002162,998613.506105
2026-06-06 02:10:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,430.0,410.0,4042.420,20.6,73.3706,6.23565,3.15253,465.098,787465.0,27.9326,1139.793007,7.190287,998613.508267,0.006975,1139.793007,1.604000,1141.397007,998613.508267,-0.004580,998613.503687
2026-06-06 02:20:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,518.0,410.0,4042.420,20.6,78.3551,6.30353,2.69654,491.876,787465.0,21.4177,1139.793007,7.484445,998613.508267,0.005348,1139.793007,1.898157,1141.691164,998613.508267,-0.006207,998613.502060
2026-06-06 02:30:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,2320.0,660.0,410.0,4042.420,20.6,80.8983,6.26082,2.97003,519.889,787465.0,12.8244,1139.793007,7.676706,998613.508267,0.003202,1139.793007,2.090419,1141.883425,998613.508267,-0.008353,998613.499914
2026-06-06 02:40:00+00:00,2730780.0,2192.0,0.0,2219678.0,2188000.0,540.0,1055.0,410.0,832.988,20.6,82.0182,6.27162,3.05909,548.217,788291.0,18.7336,1139.793007,8.013598,998613.678423,0.004678,1139.793007,2.427311,1142.220317,998613.678423,-0.006877,998613.671546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-10 09:00:00+00:00,2749540.0,2207.0,0.0,2241926.0,2188000.0,1310.0,400.0,410.0,3997.580,20.6,64.2480,6.39930,3.08181,978.709,849660.0,66.6016,1181.601248,8.202267,998626.320437,0.016630,1181.601248,2.615979,1184.217228,998626.320437,0.005075,998626.325512
2026-06-10 09:10:00+00:00,2749540.0,2207.0,0.0,2241926.0,2188000.0,1420.0,400.0,410.0,3997.580,20.6,63.4549,6.40234,3.44524,1007.620,849660.0,66.6016,1181.601248,8.249196,998626.320437,0.016630,1181.601248,2.662908,1184.264157,998626.320437,0.005075,998626.325512
2026-06-10 09:20:00+00:00,2749540.0,2207.0,0.0,2241926.0,2188000.0,1420.0,400.0,500.0,3997.580,20.6,62.3213,6.43750,3.45108,1037.660,849660.0,66.6016,1181.601248,8.281875,998626.320437,0.016630,1181.601248,2.695587,1184.296836,998626.320437,0.005075,998626.325512


In [176]:
processor.calcul_cumul_journalier(1,6,2026,"Ester")

____________________________________________________________
BILAN JOURNALIER - 01 JUIN 2026 (Terminé)
Période : du 01/06/2026 à 02:00 au 02/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.3173
  Variation de Stock     : 2.4726
  Stock Entrée.          : -0.2890
  Stock Sortie.          : 2.1836
  Production             : 10.7899
____________________________________________________________
____________________________________________________________


{'Ester': {'consommation': 8.31730239999979,
  'delta_stock': 2.472643477741661,
  'production': 10.789945877741502}}

In [177]:
dashboard.afficher_bilan_journalier(jour=7, mois=6, annee=2026)

____________________________________________________________
BILAN JOURNALIER - 07 JUIN 2026 (Terminé)
Période : du 07/06/2026 à 02:00 au 08/06/2026 à 02:00
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 9.8998
  Variation de Stock     : -0.4824
  Stock Entrée.          : 5.7741
  Stock Sortie.          : 5.2917
  Production             : 9.4174
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 3.4196
  Variation de Stock     : 0.0016
  Stock Entrée.          : -0.0003
  Stock Sortie.          : 0.0012
  Production             : 3.4212
____________________________________________________________
____________________________________________________________


In [178]:
dashboard.plot_histogramme_annuee(annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Mois         Jours    Prod Total      Prod Moy        OOE Moy      OOE Cumul   
----------------------------------------------------------------------------------------------------
January      29       187.80          6.48            68.2        % 68.2        %
February     26       186.70          7.18            75.6        % 75.6        %
March        28       217.51          7.77            81.8        % 81.8        %
April        30       246.96          8.23            86.7        % 86.7        %
May          31       261.32          8.43            88.7        % 88.7        %
June         10       92.82           9.28            97.7        % 97.7        %



In [179]:
dashboard.plot_histogramme_annee_complet(annea=2026, nom_produit='Ester')

____________________________________________________________
BILAN JOURNALIER - 01 JANVIER 2026 (Terminé)
Période : du 01/01/2026 à 02:00 au 02/01/2026 à 01:30
____________________________________________________________
PRODUIT : ESTER
------------------------------------------------------------
  Consommation (Ester)   : 8.2673
  Variation de Stock     : -4.1001
  Stock Entrée.          : 3.0225
  Stock Sortie.          : -1.0776
  Production             : 4.1672
____________________________________________________________
PRODUIT : RETINOL
------------------------------------------------------------
  Consommation (Ester)   : 2.5713
  Variation de Stock     : -0.0014
  Stock Entrée.          : 0.0015
  Stock Sortie.          : 0.0001
  Production             : 2.5700
____________________________________________________________
____________________________________________________________
____________________________________________________________
BILAN JOURNALIER - 02 JANVIER 2026 (


  📊 RÉSUMÉ ANNUEL 2026 - Ester
Nombre de jours avec données:       154/365
Production totale année:            1193.11
Production moyenne (jours actifs):  7.75
Production min/max:                 0.03 / 14.13
Écart-type:                         2.78

🎯 RATIO OOE:
  CMJ (Cible Journalière):            9.50
  OOE Moyen (par jour):               81.6%
  OOE Cumulé à date (fin d'année):    34.4%
  Jours > 100% (surproduction):       34 / 154



In [180]:
processor.plot_simple_tag('Hexane')